In [ ]:
# ── Colab Setup ──────────────────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/MScProject'
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
    os.chdir(PROJECT_ROOT)
    print(f"Working directory: {os.getcwd()}")
else:
    src_dir = os.path.join(os.getcwd(), 'src')
    if os.path.isdir(src_dir) and src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    print("Running locally.")

# 04 — Attention Head Analysis

Investigates which attention heads correspond to grammatical rules of the artificial language.

Supports three model types:
- **Custom Transformer** checkpoint (`.pt` file from `02_train.ipynb`)
- **BERT** fine-tuned checkpoint (directory from `02_train.ipynb` with `MODEL_TYPE='bert'`)
- **GPT-2** fine-tuned checkpoint (directory from `02_train.ipynb` with `MODEL_TYPE='gpt2'`)

Barry's core question: *do particular attention heads correspond to particular rules of the grammar?*

For **aⁿbⁿ** the key structural dependency is cross-serial: count n 'a' tokens then produce
exactly n 'b' tokens. We look for heads where 'b'-phase positions attend strongly back to
'a'-phase positions — a head implementing this pattern is encoding the counting rule.

**Analyses:**
1. **Attention heatmaps** — qualitative visual inspection per head and layer
2. **Cross-phase attention score** — how much do b-tokens attend to a-tokens?
3. **Attention entropy** — does a head spread attention diffusely or focus sharply?

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from torch.utils.data import DataLoader, Dataset

from grammar_loader import load_grammar, build_vocab, tokenize


# ── Model classes (self-contained copy) ─────────────────────────────────────

class GrammarDataset(Dataset):
    def __init__(self, filepath, vocab):
        self.sequences = []
        with open(filepath) as f:
            for line in f:
                line = line.strip()
                if line:
                    ids = tokenize(line, vocab, add_special=True)
                    self.sequences.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self):  return len(self.sequences)
    def __getitem__(self, idx): return self.sequences[idx]


def collate_fn(batch, pad_id):
    max_len = max(s.size(0) for s in batch)
    out = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    for i, s in enumerate(batch):
        out[i, :s.size(0)] = s
    return out


class TransformerLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=4, num_layers=2,
                 ff_dim=256, dropout=0.1, max_seq_len=512, causal_mask=True):
        super().__init__()
        self.causal_mask  = causal_mask
        self.embedding    = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_seq_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True
        )
        self.transformer  = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj  = nn.Linear(embed_dim, vocab_size)
        self.num_layers   = num_layers

    def forward(self, x):
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0)
        emb  = self.embedding(x) + self.pos_embedding(pos)
        pad_mask = (x == 0)
        causal   = None
        if self.causal_mask:
            causal = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        out = self.transformer(emb, mask=causal, src_key_padding_mask=pad_mask)
        return self.output_proj(out)

## Configuration

Point `CHECKPOINT` at a **Transformer** checkpoint saved by `02_train.ipynb`.
Run notebook 02 with `MODEL_TYPE = 'transformer'` first if you haven't already.

`EXAMPLE_N` controls which sequence lengths are used for the heatmap visualisations.

In [ ]:
# ── Checkpoint ────────────────────────────────────────────────────────────────
# For custom Transformer: path to a .pt file, e.g.:
#   CHECKPOINT = 'checkpoints/anbn_transformer_no_causal.pt'
# For BERT or GPT-2: path to the saved directory, e.g.:
#   CHECKPOINT = 'checkpoints/anbn_bert'
#   CHECKPOINT = 'checkpoints/anbn_gpt2'
CHECKPOINT   = 'checkpoints/anbn_bert'

GRAMMAR_FILE = 'grammars/anbn.txt'
DATA_FILE    = 'data/anbn_n1-100.txt'
FIGURES_DIR  = 'figures'

EXAMPLE_N    = [5, 10, 20]   # n values to visualise attention heatmaps for
BATCH_SIZE   = 32

## Load checkpoint

In [ ]:
import os, json
from pathlib import Path

device = torch.device('cpu')

LOAD_OK = True   # set to False on any config error so later cells skip gracefully

# ── Detect checkpoint type ───────────────────────────────────────────────────
IS_HF = os.path.isdir(CHECKPOINT)   # HuggingFace checkpoint = directory

if IS_HF:
    meta_path = os.path.join(CHECKPOINT, 'meta.json')
    if not os.path.exists(meta_path):
        print(f"⚠️  No meta.json found in '{CHECKPOINT}'. Is this a valid HF checkpoint?")
        LOAD_OK = False
    else:
        with open(meta_path) as f:
            meta = json.load(f)
        grammar_name = meta['grammar_name']
        model_type   = meta['model_type']    # 'bert' or 'gpt2'

        from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForCausalLM

        hf_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
        if model_type == 'bert':
            hf_model = AutoModelForMaskedLM.from_pretrained(CHECKPOINT)
        else:
            hf_model = AutoModelForCausalLM.from_pretrained(CHECKPOINT)
            hf_tokenizer.pad_token = hf_tokenizer.eos_token

        hf_model = hf_model.to(device)
        hf_model.eval()

        cfg        = hf_model.config
        num_layers = cfg.num_hidden_layers
        num_heads  = cfg.num_attention_heads
        print(f"Loaded HuggingFace {model_type} ({grammar_name})  "
              f"— {num_layers} layers × {num_heads} heads")

elif not CHECKPOINT.endswith('.pt'):
    print(f"⚠️  CHECKPOINT '{CHECKPOINT}' is neither a directory (HF) nor a .pt file "
          "(custom Transformer). Update the CHECKPOINT variable in the config cell.")
    LOAD_OK = False

else:
    try:
        ckpt = torch.load(CHECKPOINT, map_location=device)
    except FileNotFoundError:
        print(f"⚠️  Checkpoint file not found: '{CHECKPOINT}'. Run 02_train.ipynb first.")
        LOAD_OK = False
        ckpt    = None

    if ckpt is not None:
        if ckpt.get('model_type') != 'transformer':
            print(f"⚠️  Checkpoint model_type is '{ckpt.get('model_type')}', expected 'transformer'. "
                  "For LSTM probing use 03_probe.ipynb instead.")
            LOAD_OK = False
        else:
            grammar_name = ckpt['grammar_name']
            model_type   = 'transformer'
            vocab        = ckpt['vocab']
            model_args   = ckpt['args']
            pad_id       = vocab['[PAD]']
            id_to_tok    = {v: k for k, v in vocab.items()}

            model = TransformerLanguageModel(
                vocab_size  = len(vocab),
                embed_dim   = model_args['embed_dim'],
                num_heads   = 4,
                num_layers  = model_args['num_layers'],
                ff_dim      = model_args['hidden_dim'],
                causal_mask = ckpt['causal_mask'],
            )
            model.load_state_dict(ckpt['model_state'])
            model = model.to(device)
            model.eval()

            num_layers = model_args['num_layers']
            num_heads  = 4
            print(f"Loaded custom transformer ({grammar_name})  "
                  f"— {num_layers} layers × {num_heads} heads")

## Attention weight extraction

PyTorch's `TransformerEncoderLayer` discards attention weights internally.
We recover them by manually running each layer's `self_attn` with
`need_weights=True` (at the same inputs the layer will use), then advancing
through the full layer normally.

In [ ]:
if not LOAD_OK:
    print("⚠️  Skipping — checkpoint not loaded. Fix the CHECKPOINT path above and re-run.")
else:
    @torch.no_grad()
    def get_attention_weights(batch_or_encoding):
        """
        Extract per-layer attention weights from whichever model is loaded.

        For HuggingFace (BERT / GPT-2):
            Uses output_attentions=True — no hooks needed.
            Input: dict with 'input_ids' and 'attention_mask' tensors.

        For custom Transformer:
            Uses the manual forward-hook approach.
            Input: (B, T) token-ID tensor.

        Returns: list of length num_layers, each (B, num_heads, T, T).
        """
        if IS_HF:
            input_ids      = batch_or_encoding['input_ids'].to(device)
            attention_mask = batch_or_encoding['attention_mask'].to(device)

            outputs = hf_model(
                input_ids      = input_ids,
                attention_mask = attention_mask,
                output_attentions = True,
            )
            # outputs.attentions: tuple of (B, H, T, T), one per layer
            return [a.cpu() for a in outputs.attentions]

        else:
            # Custom Transformer — hook-based extraction
            batch = batch_or_encoding
            B, T  = batch.shape
            pos   = torch.arange(T, device=batch.device).unsqueeze(0)
            x     = model.embedding(batch) + model.pos_embedding(pos)
            pad_mask = (batch == 0)
            causal   = None
            if model.causal_mask:
                causal = nn.Transformer.generate_square_subsequent_mask(T, device=batch.device)

            attn_per_layer = []
            for layer in model.transformer.layers:
                _, weights = layer.self_attn(
                    x, x, x,
                    attn_mask            = causal,
                    key_padding_mask     = pad_mask,
                    need_weights         = True,
                    average_attn_weights = False,
                )
                attn_per_layer.append(weights.detach().cpu())
                x = layer(x, src_mask=causal, src_key_padding_mask=pad_mask)

            return attn_per_layer


    def ids_to_tokens_hf(input_ids, tokenizer):
        """Convert a 1-D tensor of HF token IDs to readable string labels."""
        return tokenizer.convert_ids_to_tokens(input_ids.tolist())


    def seq_for_n(n):
        """Build a single aⁿbⁿ sequence, tokenised for whichever model is loaded."""
        s = 'a' * n + 'b' * n
        if IS_HF:
            spaced = ' '.join(list(s))
            enc = hf_tokenizer(spaced, return_tensors='pt')
            return enc, ids_to_tokens_hf(enc['input_ids'][0], hf_tokenizer)
        else:
            from grammar_loader import tokenize as _tokenize
            ids    = _tokenize(s, vocab, add_special=True)
            tensor = torch.tensor(ids, dtype=torch.long).unsqueeze(0)
            tokens = [id_to_tok.get(i.item(), '?') for i in tensor[0]]
            return tensor, tokens

## Attention heatmaps — example sequences

For each n in `EXAMPLE_N`, plot a grid of heatmaps: rows = layers, columns = heads.

**What to look for:**
- Any head where 'b' tokens (right half) have high attention to 'a' tokens (left half)
  is a candidate for encoding the counting dependency.
- Diagonal patterns → local/positional attention.
- Columns all lit up → a 'sink' token that everything attends to.

In [ ]:
if not LOAD_OK:
    print("⚠️  Skipping heatmaps — checkpoint not loaded.")
else:
    Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

    for n in EXAMPLE_N:
        batch_input, tokens = seq_for_n(n)
        attn = get_attention_weights(batch_input)   # list of (1, H, T, T)

        T = len(tokens)
        fig, axes = plt.subplots(
            num_layers, num_heads,
            figsize=(num_heads * 3.2, num_layers * 3.0),
        )
        # Normalise axes to always be 2D
        if num_layers == 1:
            axes = [axes]
        if num_heads == 1:
            axes = [[ax] for ax in axes]

        for l in range(num_layers):
            for h in range(num_heads):
                ax  = axes[l][h]
                mat = attn[l][0, h].numpy()     # (T, T)
                im  = ax.imshow(mat, aspect='auto', cmap='Blues', vmin=0, vmax=mat.max())
                ax.set_title(f"L{l+1} H{h+1}", fontsize=9)
                ax.set_xticks(range(T))
                ax.set_yticks(range(T))
                ax.set_xticklabels(tokens, rotation=90, fontsize=7)
                ax.set_yticklabels(tokens, fontsize=7)
                if h == 0:
                    ax.set_ylabel('Query', fontsize=8)
                if l == num_layers - 1:
                    ax.set_xlabel('Key', fontsize=8)

        fig.suptitle(f"{grammar_name}  n={n}  model={model_type}  — attention weights", fontsize=11)
        fig.tight_layout()
        path = f"{FIGURES_DIR}/{grammar_name}_{model_type}_attn_n{n}.png"
        fig.savefig(path, dpi=150)
        plt.show()
        print(f"Saved: {path}")

## Cross-phase attention

Quantifies how much each head attends **across** the a→b boundary.

For every sequence, for every b-position query, we compute the total attention
weight directed to a-position keys.  Averaging over all sequences gives a score
in [0, 1] per head per layer.

A **high score** means that head uses b-positions to look back at a-positions —
exactly the behaviour needed to track the counting dependency.

In [ ]:
if not LOAD_OK:
    print("⚠️  Skipping cross-phase analysis — checkpoint not loaded.")
else:
    from torch.utils.data import DataLoader as _DL, Dataset as _DS
    from grammar_loader import load_grammar as _load_grammar

    grammar = _load_grammar(GRAMMAR_FILE)

    # Build a dataloader appropriate for the loaded model
    if IS_HF:
        class _HFDataset(_DS):
            def __init__(self, filepath, tokenizer, max_length=256):
                with open(filepath) as f:
                    strings = [l.strip() for l in f if l.strip()]
                spaced = [' '.join(list(s)) for s in strings]
                enc = tokenizer(spaced, padding='max_length', truncation=True,
                                max_length=max_length, return_tensors='pt')
                self.input_ids      = enc['input_ids']
                self.attention_mask = enc['attention_mask']
                self.strings = strings

            def __len__(self): return len(self.input_ids)
            def __getitem__(self, idx):
                return {'input_ids': self.input_ids[idx],
                        'attention_mask': self.attention_mask[idx],
                        'string': self.strings[idx]}

        hf_ds   = _HFDataset(DATA_FILE, hf_tokenizer)
        def _collate_hf(items):
            return {
                'input_ids':      torch.stack([i['input_ids']      for i in items]),
                'attention_mask': torch.stack([i['attention_mask']  for i in items]),
                'strings':        [i['string'] for i in items],
            }
        dataloader = _DL(hf_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=_collate_hf)

        def _get_phase_masks(batch):
            a_tok  = hf_tokenizer.convert_tokens_to_ids('a')
            b_tok  = hf_tokenizer.convert_tokens_to_ids('b')
            ids    = batch['input_ids']
            return (ids == a_tok), (ids == b_tok)

    else:
        class _CustomDS(_DS):
            def __init__(self, filepath, vocab, pad_id):
                from grammar_loader import tokenize as _tok
                self.seqs = []
                with open(filepath) as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            ids = _tok(line, vocab, add_special=True)
                            self.seqs.append(torch.tensor(ids, dtype=torch.long))
                self.pad_id = pad_id
            def __len__(self): return len(self.seqs)
            def __getitem__(self, idx): return self.seqs[idx]

        def _collate_custom(batch):
            max_len = max(s.size(0) for s in batch)
            out = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
            for i, s in enumerate(batch):
                out[i, :s.size(0)] = s
            return out

        custom_ds  = _CustomDS(DATA_FILE, vocab, pad_id)
        dataloader = _DL(custom_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=_collate_custom)

        def _get_phase_masks(batch):
            a_id = vocab.get('a', -1)
            b_id = vocab.get('b', -1)
            return (batch == a_id), (batch == b_id)


    # ── Cross-phase attention accumulation ──────────────────────────────────────
    cross_sum   = np.zeros((num_layers, num_heads))
    cross_count = 0

    for batch in dataloader:
        if IS_HF:
            attn = get_attention_weights(batch)
            ids  = batch['input_ids']
        else:
            attn = get_attention_weights(batch)
            ids  = batch

        a_mask, b_mask = _get_phase_masks({'input_ids': ids} if IS_HF else ids)

        for b_idx in range(ids.size(0)):
            a_pos = a_mask[b_idx]
            b_pos = b_mask[b_idx]
            if not b_pos.any() or not a_pos.any():
                continue
            for l in range(num_layers):
                w      = attn[l][b_idx]                     # (H, T, T)
                b_to_a = w[:, b_pos, :][:, :, a_pos]       # (H, #b, #a)
                cross_sum[l] += b_to_a.sum(dim=(1, 2)).numpy()
            cross_count += b_pos.sum().item()

    cross_phase_score = cross_sum / max(cross_count, 1)

    # ── Plot ─────────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, num_layers, figsize=(num_layers * 4, 3.5), sharey=True)
    if num_layers == 1:
        axes = [axes]

    for l, ax in enumerate(axes):
        scores = cross_phase_score[l]
        ax.bar(range(1, num_heads + 1), scores, color='steelblue')
        ax.set_title(f"Layer {l+1}")
        ax.set_xlabel('Head')
        if l == 0:
            ax.set_ylabel('Mean b→a attention')
        ax.set_ylim(0, max(cross_phase_score.max() * 1.2, 0.1))
        ax.set_xticks(range(1, num_heads + 1))

    fig.suptitle(f"{grammar_name} ({model_type}) — cross-phase attention (b-tokens attending to a-tokens)",
                 fontsize=10)
    fig.tight_layout()
    path = f"{FIGURES_DIR}/{grammar_name}_{model_type}_cross_phase.png"
    fig.savefig(path, dpi=150)
    plt.show()
    print(f"Saved: {path}")
    print("\nCross-phase scores (layer × head):")
    print(np.round(cross_phase_score, 4))

## Attention entropy per head

Shannon entropy of each head's attention distribution (averaged over all query
positions and sequences).

- **High entropy** → head spreads attention broadly (diffuse / positional)
- **Low entropy** → head attends sharply to specific positions (potentially rule-specific)

Heads with low entropy and high cross-phase score are the strongest candidates
for encoding the counting rule.

In [ ]:
if not LOAD_OK:
    print("⚠️  Skipping entropy analysis — checkpoint not loaded.")
else:
    entropy_sum   = np.zeros((num_layers, num_heads))
    entropy_count = 0

    for batch in dataloader:
        if IS_HF:
            attn     = get_attention_weights(batch)
            ids      = batch['input_ids']
            pad_tok  = hf_tokenizer.pad_token_id
            pad_mask = (ids == pad_tok)
        else:
            attn     = get_attention_weights(batch)
            ids      = batch
            pad_mask = (ids == pad_id)

        for b_idx in range(ids.size(0)):
            valid = ~pad_mask[b_idx]
            T_val = valid.sum().item()
            if T_val == 0:
                continue
            for l in range(num_layers):
                w       = attn[l][b_idx]           # (H, T, T)
                w_valid = w[:, valid, :]            # (H, T_val, T)
                w_clip  = w_valid.clamp(min=1e-9)
                H_val   = -(w_clip * w_clip.log()).sum(dim=-1).mean(dim=-1)   # (H,)
                entropy_sum[l] += H_val.numpy()
            entropy_count += 1

    mean_entropy = entropy_sum / max(entropy_count, 1)

    # ── Plot: entropy + cross-phase side by side ─────────────────────────────────
    fig, axes = plt.subplots(2, num_layers,
                             figsize=(max(num_layers * 4, 6), 5),
                             sharey='row')
    if num_layers == 1:
        axes = [[axes[0]], [axes[1]]]

    for l in range(num_layers):
        axes[0][l].bar(range(1, num_heads + 1), mean_entropy[l],    color='darkorange')
        axes[0][l].set_title(f"Layer {l+1} — entropy",    fontsize=9)
        axes[0][l].set_xticks(range(1, num_heads + 1))
        if l == 0:
            axes[0][l].set_ylabel('Mean attention entropy')

        axes[1][l].bar(range(1, num_heads + 1), cross_phase_score[l], color='steelblue')
        axes[1][l].set_title(f"Layer {l+1} — cross-phase", fontsize=9)
        axes[1][l].set_xlabel('Head')
        axes[1][l].set_xticks(range(1, num_heads + 1))
        if l == 0:
            axes[1][l].set_ylabel('Cross-phase score')

    fig.suptitle(f"{grammar_name} ({model_type}) — entropy (top) and cross-phase score (bottom)",
                 fontsize=10)
    fig.tight_layout()
    path = f"{FIGURES_DIR}/{grammar_name}_{model_type}_entropy.png"
    fig.savefig(path, dpi=150)
    plt.show()
    print(f"Saved: {path}")
    print("\nMean entropy (layer × head):")
    print(np.round(mean_entropy, 4))

## Interpretation guide

| Pattern | Interpretation |
|---------|----------------|
| High cross-phase + low entropy | Strong candidate for encoding the counting rule |
| Low cross-phase + low entropy  | Head attends to specific positions but within one phase (e.g. local context) |
| High entropy                   | Diffuse / positional head; less likely to be rule-specific |

Cross-reference with the heatmaps: a head with high cross-phase score should show
a clear off-diagonal block in the heatmap (b-row × a-column lit up).

Repeat this notebook with `CHECKPOINT` pointing at an **anbncn** transformer
checkpoint to compare how a context-sensitive grammar is represented.